In [1]:
import pandas as pd

wine_source = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv"
wine = pd.read_csv(wine_source, sep=";")

X = wine.drop(columns=["quality"])
y = wine["quality"]

print(X.shape)
print(y.shape)

(4898, 11)
(4898,)


1. Split traiing/val/test

In [2]:
from sklearn.model_selection import train_test_split

# reserve 20% for testing
X_remaining, X_test, y_remaining, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
)

# take 10% of the original rows from the remaining 80%
X_train, X_val, y_train, y_val = train_test_split(
    X_remaining, y_remaining,
    test_size=0.10 / 0.80,
    random_state=42,
)

print("train count: ", len(X_train))
print("val count: ", len(X_val))
print("Test count: ", len(X_test))

train count:  3428
val count:  490
Test count:  980


2. build knn grid

In [11]:
from sklearn.model_selection import ParameterGrid

# vary neighbor count, distance metric, and weighting
knn_options = {
    "n_neighbors": range(1, 128),
    "metric": ["euclidean", "manhattan"],
    "weights": ["uniform", "distance"],
}

# Form all 500 combinations
knn_grid = list(ParameterGrid(knn_options))

print("Candidate settings:", len(knn_grid))
print("Validation wines:", len(y_val))

Candidate settings: 508
Validation wines: 490


3. choose best knn w.r.t. val MSE

In [12]:
from sklearn.metrics import mean_squared_error
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler

# learn feature scales from training wines only
# scale all teh features, we have to fit a model
# rather than regular noramlizing because we want
# account for each feature's own std deviation
# THAT is how you get noramlized comparisons
knn_scaler = StandardScaler()

# learn each feature's mean + std dev
# (fit_transform() = fit()->transform())
X_train_scaled = knn_scaler.fit_transform(X_train)
X_val_scaled = knn_scaler.transform(X_val)

# vars to track the lowest validation MSE
best_knn = None
best_knn_params = None
best_knn_val_mse = float("inf")

for params in knn_grid:
    # fit knn THIS model
    model = KNeighborsRegressor(**params)
    model.fit(X_train_scaled, y_train)

    # pred / mse
    val_predictions = model.predict(X_val_scaled)
    val_mse = mean_squared_error(y_val, val_predictions)

    # choose the best mse model based on val score
    if val_mse < best_knn_val_mse:
        best_knn = model
        best_knn_params = params
        best_knn_val_mse = val_mse

print("Best KNN settings:", best_knn_params)
print("Best validation MSE:", best_knn_val_mse)

Best KNN settings: {'metric': 'manhattan', 'n_neighbors': 25, 'weights': 'distance'}
Best validation MSE: 0.2987389892437149


4. random forest grid

In [7]:
# vary tree depth, min leaf size, and features per split
rf_options = {
    "n_estimators": [10, 100, 300],
    "max_depth": [4, 8, 16, 32, None],
    "min_samples_leaf": range(1, 10),
    "max_features": range(1, 10),
}

# form all combinations of the above
rf_grid = list(ParameterGrid(rf_options))

print("Candidate settings:", len(rf_grid))
print("Validation wines:", len(y_val))

Candidate settings: 1215
Validation wines: 490


5. chose best rf w.r.t. val MSE

In [8]:
from sklearn.ensemble import RandomForestRegressor

# trackers for best validation mse
best_rf = None
best_rf_params = None
best_rf_val_mse = float("inf")

# for each param of the XXX combinations
# train and test the generated tree
for params in rf_grid:
    model = RandomForestRegressor(
        random_state=42,
        **params,
    )
    model.fit(X_train, y_train)

    # pred / mse
    val_predictions = model.predict(X_val)
    val_mse = mean_squared_error(y_val, val_predictions)

    # keep the best
    if val_mse < best_rf_val_mse:
        best_rf = model
        best_rf_params = params
        best_rf_val_mse = val_mse

print("Best random-forest settings:", best_rf_params)
print("Best validation MSE:", best_rf_val_mse)

Best random-forest settings: {'max_depth': 32, 'max_features': 3, 'min_samples_leaf': 1, 'n_estimators': 100}
Best validation MSE: 0.30249072978851405


6. eval both on test set

In [9]:
# use knn *training-data* scaler on test
X_test_scaled = knn_scaler.transform(X_test)

knn_test_predictions = best_knn.predict(X_test_scaled)
knn_test_mse = mean_squared_error(y_test, knn_test_predictions)

# pred/mse the forest on the same wines
# in their *original* units
rf_test_predictions = best_rf.predict(X_test)
rf_test_mse = mean_squared_error(y_test, rf_test_predictions)

print("KNN test MSE:", knn_test_mse)
print("Random-forest test MSE:", rf_test_mse)

KNN test MSE: 0.36855023327473463
Random-forest test MSE: 0.3525888155398214


7. cmp val and test err

In [13]:
results = pd.DataFrame({
    "Model": ["KNN", "Random forest"],
    "Validation MSE": [best_knn_val_mse, best_rf_val_mse],
    "Test MSE": [knn_test_mse, rf_test_mse],
})

results["Test minus validation"] = results["Test MSE"] - results["Validation MSE"]

print(results.round(5))

           Model  Validation MSE  Test MSE  Test minus validation
0            KNN         0.29874   0.36855                0.06981
1  Random forest         0.30249   0.35259                0.05010
